In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd
import numpy as np
import re
from tqdm import tqdm

# ==============================
# 1️⃣ Load Data
# ==============================

speech_df = pd.read_csv("/kaggle/input/datasets/avalonw/the-feds-public-speech-transcript/fed_speech.csv")

# Clean text
def clean_text(text):
    text = str(text)
    text = text.replace("\n", " ")
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

speech_df["content"] = speech_df["content"].apply(clean_text)

# Remove empty speeches
speech_df = speech_df[speech_df["content"].str.len() > 0].reset_index(drop=True)

# ==============================
# 2️⃣ Load FinBERT
# ==============================

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModel.from_pretrained("ProsusAI/finbert")

model.eval()

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Using device:", device)

# ==============================
# 3️⃣ Speech Embedding Function
# ==============================

def speech_embedding(text, max_length=512, stride=50, chunk_batch_size=8):
    # Tokenize with overflow
    tokens = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        padding=True
    )

    input_ids = tokens["input_ids"]
    attention_mask = tokens["attention_mask"]
    
    num_chunks = input_ids.size(0)
    all_cls_embeddings = []

    # Process chunks in smaller batches to save VRAM
    for i in range(0, num_chunks, chunk_batch_size):
        batch_input_ids = input_ids[i : i + chunk_batch_size].to(device)
        batch_attention_mask = attention_mask[i : i + chunk_batch_size].to(device)

        with torch.no_grad():
            outputs = model(
                input_ids=batch_input_ids,
                attention_mask=batch_attention_mask
            )
        
        # Pull CLS tokens and move to CPU immediately to free GPU memory
        cls_batch = outputs.last_hidden_state[:, 0, :].cpu()
        all_cls_embeddings.append(cls_batch)
        
        # Clear cache to be safe
        del outputs
        # torch.cuda.empty_cache() # Optional: use if it still crashes

    # Combine all chunks and calculate mean
    combined_cls = torch.cat(all_cls_embeddings, dim=0)
    speech_embed = torch.mean(combined_cls, dim=0)

    return speech_embed.numpy()
    
# ==============================
# 4️⃣ Generate All Embeddings
# ==============================

all_embeddings = []

for text in tqdm(speech_df["content"]):
    emb = speech_embedding(text)
    all_embeddings.append(emb)

embeddings = np.vstack(all_embeddings)

print("Embedding shape:", embeddings.shape)

# ==============================
# 5️⃣ Save Embeddings
# ==============================

np.save("/kaggle/working/fed_speech_embeddings.npy", embeddings)
print("Done.")